## Importing Libraries

In [22]:
import janitor
import skimpy as sk
import duckdb as db 
import pandas as pd
import plotly.express as px

In [23]:
import warnings as w

# Ignore all warnings
w.filterwarnings("ignore")

## Loading the Dataset

In [ ]:
df = pd.read_csv("Project01/sales-data-analysis-main/sales_data.csv")

## Data Overview

In [25]:
df.head()

,date,product,category,price,quantity,revenue
0,2022-01-01,Smartphone,Electronics,600.0,10.0,6000.0
1,2022-01-01,Laptop,Electronics,1200.0,5.0,6000.0
2,2022-01-02,T-Shirt,Clothing,20.0,50.0,1000.0
3,2022-01-03,Headphones,Electronics,100.0,20.0,2000.0
4,2022-01-04,T-Shirt,Clothing,20.0,25.0,500.0


In [26]:
import warnings as w 
sk.skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 369    │ │ string      │ 3     │                                                          │
│ │ Number of columns │ 6      │ │ float64     │ 3     │                                                          │
│ └───────────────────┴────────┘ └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━┳━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━┓  │
│ ┃ column     ┃ NA  ┃ NA %                   ┃ mean   ┃ sd     ┃ p0   ┃ p25  ┃ p50   ┃ p75   ┃ p100  ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━╇━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━┩  │
│ │   price    │   2 │     0.5420054200542005 │  211.2 │  227.3 │   20 │   50 │   100 │   300 │  1200 │  █ ▂   │  │
│ │  quantity  │   1 │    0.27100271002710025 │  14.57 │  8.596 │    3 │    8 │    12 │    20 │    50 │ █▅▃▁▁  │  │
│ │  revenue   │   1 │    0.27100271002710025 │   2061 │   1911 │  300 │  800 │  1200 │  2400 │  7200 │ █▃ ▁▁▁ │  │
│ └────────────┴─────┴────────────────────────┴────────┴────────┴──────┴──────┴───────┴───────┴───────┴────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓  │
│ ┃          ┃    ┃      ┃           ┃           ┃           ┃            ┃ chars per ┃ words per  ┃ total     ┃  │
│ ┃ column   ┃ NA ┃ NA % ┃ shortest  ┃ longest   ┃ min       ┃ max        ┃ row       ┃ row        ┃ words     ┃  │
│ ┡━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩  │
│ │   date   │  0 │    0 │ 2022-01-0 │ 2022-01-0 │ 2022-01-0 │ 2022-12-31 │        10 │          1 │       369 │  │
│ │          │    │      │     1     │     1     │     1     │            │           │            │           │  │
│ │ product  │  0 │    0 │   Coat    │ Smartphon │ Backpack  │   Watch    │      7.54 │          1 │       369 │  │
│ │          │    │      │           │     e     │           │            │           │            │           │  │
│ │ category │  0 │    0 │   Bags    │ Electroni │ Accessori │  Shoeses   │      9.18 │          1 │       369 │  │
│ │          │    │      │           │    cs     │    es     │            │           │            │           │  │
│ └──────────┴────┴──────┴───────────┴───────────┴───────────┴────────────┴───────────┴────────────┴───────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

##  Data Quality Summary (Skimpy Output)

Skimpy summary indicates 3 data issues:

1) Date is saved as a string, this will need to be transformed for time-series analysis  

2) Missing Data in numerical columns (the percentage of missing data is small so we can leave it but i will investigate to identify if we can't resolve the missing data)  

3) Inconsistent Naming Scheme in Categories (Need to assess how many rows are affected)

## Data Preprocessing & Overview

In [27]:
df['date'] = pd.to_datetime(df['date'])

In [28]:
db.query("Select distinct category from df")

┌─────────────┐
│  category   │
│   varchar   │
├─────────────┤
│ Shoes       │
│ Shoeses     │
│ Clothing    │
│ Accessories │
│ Electronics │
│ Bags        │
│ Clohting    │
│ Bgas        │
└─────────────┘

In [29]:
df.loc[df['category'] == 'Bgas', 'category'] = 'Bags'
df.loc[df['category'] == 'Clohting', 'category'] = 'Clothing'
df.loc[df['category'] == 'Shoeses','category'] = 'Shoes'

In [30]:
db.query("Select distinct product from df")

┌────────────┐
│  product   │
│  varchar   │
├────────────┤
│ T-Shirt    │
│ Headphones │
│ Watch      │
│ Hoodie     │
│ Coat       │
│ Speaker    │
│ Sneakers   │
│ Laptop     │
│ Backpack   │
│ Wallet     │
│ Smartphone │
│ Tablet     │
│ Smartwatch │
│ Jeans      │
└────────────┘
   14 rows  

In [31]:
df.loc[df['product'] == 'Bgas', 'product'] = 'Bags'

#### revenue = (price*quantity)

In [32]:
db.query("Select * from df where price is null or quantity is null or revenue is null")

┌─────────────────────┬────────────┬─────────────┬────────┬──────────┬─────────┐
│        date         │  product   │  category   │ price  │ quantity │ revenue │
│      timestamp      │  varchar   │   varchar   │ double │  double  │ double  │
├─────────────────────┼────────────┼─────────────┼────────┼──────────┼─────────┤
│ 2022-04-05 00:00:00 │ Smartwatch │ Accessories │  200.0 │     10.0 │    NULL │
│ 2022-05-01 00:00:00 │ Smartphone │ Electronics │  600.0 │     NULL │  6600.0 │
│ 2022-07-11 00:00:00 │ Watch      │ Accessories │   NULL │     15.0 │  2250.0 │
│ 2022-11-13 00:00:00 │ Wallet     │ Accessories │   NULL │     35.0 │  1050.0 │
└─────────────────────┴────────────┴─────────────┴────────┴──────────┴─────────┘

In [33]:
df['price'] = df['price'].fillna(df['revenue'] / df['quantity'])
df['quantity'] = df['quantity'].fillna(df['revenue'] / df['price'])
df['revenue'] = df['revenue'].fillna(df['price'] * df['quantity'])

In [34]:
sk.skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 369    │ │ float64     │ 3     │                                                          │
│ │ Number of columns │ 6      │ │ string      │ 2     │                                                          │
│ └───────────────────┴────────┘ │ datetime64  │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓  │
│ ┃ column       ┃ NA   ┃ NA %    ┃ mean     ┃ sd       ┃ p0    ┃ p25   ┃ p50     ┃ p75    ┃ p100   ┃ hist     ┃  │
│ ┡━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩  │
│ │    price     │    0 │       0 │    210.6 │    226.9 │    20 │    50 │     100 │    200 │   1200 │   █ ▂    │  │
│ │   quantity   │    0 │       0 │    14.56 │    8.586 │     3 │     8 │      12 │     20 │     50 │  █▅▃▁▁   │  │
│ │   revenue    │    0 │       0 │     2061 │     1908 │   300 │   800 │    1200 │   2400 │   7200 │  █▃ ▁▁▁  │  │
│ └──────────────┴──────┴─────────┴──────────┴──────────┴───────┴───────┴─────────┴────────┴────────┴──────────┘  │
│                                                    datetime                                                     │
│ ┏━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column         ┃ NA     ┃ NA %       ┃ first                 ┃ last                  ┃ frequency           ┃  │
│ ┡━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩  │
│ │      date      │      0 │          0 │      2022-01-01       │      2022-12-31       │ None                │  │
│ └────────────────┴────────┴────────────┴───────────────────────┴───────────────────────┴─────────────────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┓  │
│ ┃          ┃    ┃      ┃          ┃            ┃             ┃       ┃ chars per  ┃ words per   ┃ total      ┃  │
│ ┃ column   ┃ NA ┃ NA % ┃ shortest ┃ longest    ┃ min         ┃ max   ┃ row        ┃ row         ┃ words      ┃  │
│ ┡━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━┩  │
│ │ product  │  0 │    0 │   Coat   │ Smartphone │  Backpack   │ Watch │       7.54 │           1 │        369 │  │
│ │ category │  0 │    0 │   Bags   │ Electronic │ Accessories │ Shoes │       9.17 │           1 │        369 │  │
│ │          │    │      │          │     s      │             │       │            │             │            │  │
│ └──────────┴────┴──────┴──────────┴────────────┴─────────────┴───────┴────────────┴─────────────┴────────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

# Assessment Questions

1. What was the total revenue generated by the company over the course of the year?

2. Which product had the highest revenue? How much revenue did it generate?

3. What was the average price of a product sold by the company?

4. What was the total quantity of products sold by the company?

5. Which category had the highest revenue? How much revenue did it generate?

6. What was the average revenue per sale?

7. What was the total revenue generated in each quarter of the year?
   - Q1
   - Q2
   - Q3
   - Q4

## Analysis Structure & Approach

After analysing the questions, I identified that they fall into three main analytical categories:

- **Descriptive Analytics** – calculating averages and totals for each order  
- **Comparative Analysis** – identifying which category/product performed best  
- **Trend Analysis** – evaluating quarterly performance over time  

---

Based on this, I structured the analysis into four key outputs:

**A)** A summary table showing average and total price and revenue  

**B)** Bar charts to compare category and product performance and identify top performers  

**C)** A line chart to visualize revenue fluctuations across the year  


In [35]:
summary = db.query("""-- Overall Sales Performance Summary (DuckDB)

WITH summary AS (
    SELECT
        ROUND(AVG(price), 0)      AS avg_price,
        ROUND(AVG(revenue), 0)    AS avg_revenue,
        SUM(quantity)             AS total_quantity,
        ROUND(SUM(revenue), 0)    AS total_revenue
    FROM df
)

SELECT 'avg_price' AS metric,
       '$' || CAST(avg_price AS BIGINT) AS value
FROM summary

UNION ALL

SELECT 'avg_revenue' AS metric,
       '$' || CAST(avg_revenue AS BIGINT) AS value
FROM summary

UNION ALL

SELECT 'total_quantity' AS metric,
       CAST(total_quantity AS BIGINT) AS value
FROM summary

UNION ALL

SELECT 'total_revenue' AS metric,
       '$' || CAST(total_revenue AS BIGINT) AS value
FROM summary;""")

summary

┌────────────────┬─────────┐
│     metric     │  value  │
│    varchar     │ varchar │
├────────────────┼─────────┤
│ avg_price      │ $211    │
│ avg_revenue    │ $2061   │
│ total_quantity │ 5371    │
│ total_revenue  │ $760330 │
└────────────────┴─────────┘

In [36]:
rev = db.query("""
    SELECT product,
           SUM(revenue) AS total_revenue
    FROM df
    GROUP BY product
""").df()

# Ensure correct sorting (highest → lowest)
rev = rev.sort_values("total_revenue", ascending=False)

# Color logic
rev["color"] = ["lightblue" if i == 0 else "darkgray" for i in range(len(rev))]

fig = px.bar(
    rev,
    y="product",
    x="total_revenue",
    color="color",
    color_discrete_map="identity",
    template="simple_white",
    title="Total Revenue Per Product"
)

fig.update_layout(
    title_x=0,

    xaxis_title="Total Revenue",
    xaxis=dict(title_standoff=10),

    yaxis=dict(
        categoryorder="array",
        categoryarray=rev["product"],
        autorange="reversed"
    ),

    showlegend=False,
    margin=dict(l=80)
)

fig.show()

## Key Insight: Product Dominance

There is a clear dominance of smart devices such as phones and watches in the dataset. 

Product comparison shows that smartphones are the leading product, indicating that they are a key revenue driver.

However, this dominance also creates a skewed perception of overall performance, as the remaining products appear significantly smaller in comparison.

The other product categories perform relatively modestly, each sitting between 10K and 50K in total revenue, while smartwatches perform slightly above this range.

In [37]:

# Query category revenue
cat_rev = db.query("""
    SELECT category, ROUND(SUM(revenue), 2) AS total_revenue
    FROM df
    GROUP BY category
    ORDER BY total_revenue DESC
""").df()

# Ensure correct sorting (highest → lowest)
cat_rev = cat_rev.sort_values("total_revenue", ascending=False)

# Color logic: highlight highest category
cat_rev["color"] = ["lightblue" if i == 0 else "darkgray" for i in range(len(cat_rev))]

fig = px.bar(
    cat_rev,
    y="category",
    x="total_revenue",
    color="color",
    color_discrete_map="identity",
    template="simple_white",
    title="Total Revenue Per Category"
)

fig.update_layout(
    title_x=0,  # left-aligned title

    xaxis_title="Total Revenue",
    xaxis=dict(title_standoff=10),

    yaxis=dict(
        categoryorder="array",
        categoryarray=cat_rev["category"],
        autorange="reversed"  # highest at top
    ),

    showlegend=False,
    margin=dict(l=90)
)

# --- Add value label for top category only ---
top_category = cat_rev.iloc[0]

fig.add_annotation(
    x=top_category["total_revenue"],
    y=top_category["category"],
    text=f"{top_category['total_revenue']:,.2f}",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(size=12, color="black"),
    bgcolor="rgba(255,255,255,0.8)"
)

fig.show()

## Revenue by Category Insight

Revenue by category shows that Electronics is the most profitable segment. This is a direct result of the dominance of smart devices within total revenue share.

Overall category performance suggests that the number of products within a category is a key driver when analyzing profitability, as categories with more high-performing products tend to generate significantly higher revenue.

In [38]:
import plotly.express as px

# Query quarterly revenue
rev_trend = db.query("""
    SELECT Quarter(date) AS yearly_quarter,
           SUM(revenue) AS total_revenue
    FROM df
    GROUP BY Quarter(date)
    ORDER BY yearly_quarter
""").df()

# Create line chart with markers
fig = px.line(
    rev_trend,
    x="yearly_quarter",
    y="total_revenue",
    markers=True,
    title="Revenue Per Quarter",
    template="simple_white"
)

# Fix x-axis to quarters 1–4
fig.update_layout(
    title_x=0,
    xaxis=dict(
        tickmode="array",
        tickvals=[1, 2, 3, 4],
        ticktext=["Q1", "Q2", "Q3", "Q4"],
        title="Quarter"
    ),
    yaxis_title="Total Revenue"
)

# Add elevated, underlined revenue labels
for _, row in rev_trend.iterrows():
    fig.add_annotation(
        x=row["yearly_quarter"],
        y=row["total_revenue"],
        text=f"<u>{row['total_revenue']:,.2f}</u>",
        showarrow=False,
        yshift=26,  # elevated to avoid overlap
        font=dict(size=12, color="black")
    )

fig.show()

##  Quarterly Revenue Trend Insight

Sales figures increased steadily from the first quarter, peaking at nearly 198,000 in the third quarter. This was followed by a slight decline in the fourth quarter, with revenue capping at almost 195,000.

Although this decrease is relatively small, it may indicate a softening in demand, particularly within the electronics category, which remains the primary driver of overall revenue.